# Amazon ML Challenge — Business Entity Resolution
## Master Orchestration Notebook

This master notebook serves as the execution orchestrator across all pipeline stages.
It connects Google Drive storage, clones/pulls the source repository from GitHub, verifies the runtime, and executes pipeline stages independently.

### Step 1: Environment Setup & Repository Synchronization

In [ ]:
# 1. Environment & GPU Check
import os
import sys
import subprocess
from pathlib import Path

# Set repository settings
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/amazon-ml-entity-resolution.git"  # Update with your GitHub repo URL
BRANCH = "main"

# Mount Google Drive if running in Colab/Kaggle with Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully at /content/drive")
except Exception:
    print("Not in Colab or Google Drive already accessible.")

# Ensure project root is in working directory
WORKING_DIR = Path(".").resolve()
print(f"Current working directory: {WORKING_DIR}")

In [ ]:
# 2. Pull latest code from GitHub
if Path(".git").exists():
    print("Git repository detected. Pulling latest updates...")
    !git fetch origin && git checkout main && git pull
else:
    print(f"Cloning repository from {REPO_URL}...")
    # !git clone {REPO_URL} .

# Add current directory to Python module search path
if str(WORKING_DIR) not in sys.path:
    sys.path.insert(0, str(WORKING_DIR))

# Print Git Commit Hash
!git rev-parse HEAD

In [ ]:
# 3. Install Pinned Dependencies
!pip install -q -r requirements.txt

### Step 2: Storage & Dataset Access Verification

In [ ]:
from src.utils.storage import StorageManager

storage = StorageManager("configs/config.yaml")
storage.initialize_directories()

print("Storage Summary:")
print(f" - Master Dataset Zip : {storage.gdrive_zip_path}")
print(f" - Local Raw Dir       : {storage.raw_dir.resolve()}")
print(f" - Processed Dir       : {storage.processed_dir.resolve()}")
print(f" - Checkpoints Dir     : {storage.checkpoints_dir.resolve()}")
print(f" - Output Dir          : {storage.output_dir.resolve()}")

# Check if dataset is available or sync from zip
has_data = storage.has_raw_dataset()
print(f"Raw dataset ready: {has_data}")
if not has_data:
    print(f"Attempting to extract master zip from {storage.gdrive_zip_path}...")
    storage.sync_dataset_from_zip()

### Step 3: Stage Execution Control

Set the desired `STAGE_TO_RUN` below and execute the cell.
Supported stages:
- `data_audit`
- `eda`
- `preprocessing`
- `validation`
- `blocking`
- `candidate_recall`
- `features`
- `baseline`
- `training`
- `threshold_optimization`
- `advanced_modeling`
- `hard_negatives`
- `model_selection`
- `test_inference`
- `submission`
- `reproducibility_package`

In [ ]:
# Set stage name to execute
STAGE_TO_RUN = "data_audit"  # Change to the desired stage
FORCE_RECOMPUTE = False       # Set True to bypass caching

# Run stage via StageController
from src.utils.stage_runner import StageController

controller = StageController("configs/config.yaml")
success = controller.run_stage(stage_name=STAGE_TO_RUN, force=FORCE_RECOMPUTE)

print(f"Stage '{STAGE_TO_RUN}' execution status: {'SUCCESS' if success else 'FAILED'}")